# `live_mixing` — User Guide

`live_mixing` reads, joins, and exports data from a [DJUCED](https://www.hercules.com/en-us/product/djuced/)
DJ software SQLite database (`djuced.db`). DJUCED keeps no session/play-log — this package's one
piece of real design logic is reconstructing DJ sessions and setlists from `tracks.last_played`
timestamps, and pairing them with DJUCED's recorded-mix audio files.

```
live_mixing/
├── __init__.py            re-exports every public function below
└── read_djuced_db.py      all logic: reads, joins, session reconstruction, CSV export
```

## What each function needs

| Function | Needs DB file | Notes |
|---|---|---|
| `read_djuced_db` | yes | base helper — every other `read_*` goes through it |
| `read_djuced_playlists` | yes | raw `playlists2` table |
| `read_djuced_playlist_tracks` | yes | `playlists2` ⋈ `tracks` |
| `read_track_cues` | yes | `trackCues` ⋈ `tracks` |
| `read_track_beatgrid` | yes | `trackBeats` ⋈ `tracks` |
| `read_djuced_session` | yes | tracks played in a timestamp range |
| `list_sessions` | yes | auto-detects session boundaries |
| `current_track` | yes | single most-recently-played track ("now playing") |
| `match_recording_to_session` | yes | pairs `recordings` rows to sessions |
| `snapshot_play_log` | yes + reads/writes its own `log_dir` | diffs playcount vs. the last snapshot, appends new plays to a persistent log |
| `find_missing_files` | yes + disk access | stats every track path |
| `top_played_tracks` | yes | |
| `export_*_csv` (5 functions) | yes + disk write | writes under `data/` |

None of these need network access or an API key — the only real dependency is the SQLite file
and `pandas`.

## Prerequisites

```
pip install -e .
```

**Run this notebook top-to-bottom** — use *Kernel → Restart & Run All*, or Shift+Enter through
every cell in order without skipping any. `db_path` is resolved once in Section 1 (Sample data)
and reused by every cell after it; jumping straight to a later section without running Section 1
first will raise `NameError: name 'db_path' is not defined`.

## Section index

1. Setup & sample data
2. Core reader — `read_djuced_db`
3. Playlists — `read_djuced_playlists`, `read_djuced_playlist_tracks`
4. Track detail — `read_track_cues`, `read_track_beatgrid`
5. Session reconstruction — `read_djuced_session`, `list_sessions`, `current_track`
6. Recording ↔ session matching — `match_recording_to_session`
7. Play-log tracking — `snapshot_play_log`
8. Library maintenance — `find_missing_files`, `top_played_tracks`
9. CSV export — all five `export_*_csv` functions
10. End-to-end pattern
11. Error handling reference


In [ ]:
import os
import shutil
import sqlite3
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

import live_mixing as lm

# ---- Tunables ----
GAP_MINUTES = 15        # session-boundary gap threshold, see list_sessions()
RUN_FULL = False         # flip to True to export the full tracks/playlists tables (slower)
OUTPUT_DIR = REPO_ROOT / "data"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Repo root   : {REPO_ROOT}")
print(f"Default DB  : {lm.DEFAULT_DB_PATH}")
print(f"Output dir  : {OUTPUT_DIR}")
print("Setup OK")


## 1. Sample data

Every function needs a real `djuced.db` SQLite file. If one exists at the default location
(`~/Documents/DJUCED/djuced.db`, overridable via `db_path=` on every function) this guide runs
against it directly. Otherwise it builds a small **synthetic** database with the same schema —
`tracks`, `playlists2`, `trackCues`, `trackBeats`, `recordings` — so every cell below still runs.

The synthetic fixture deliberately includes:
- a short "library browsing" cluster (plays a few seconds apart — too fast to be real mixing)
- a real 6-track mixing session (plays ~4 minutes apart)
- a decoy session on a different date that starts at a *similar time of day* but has a very
  different duration
- a recording whose filename time-of-day is close to **both** candidate sessions' start times, so
  `match_recording_to_session`'s duration-based disambiguation (see `docs/architecture.md`) has
  something real to resolve
- one track pointing at a file that doesn't exist on disk, for `find_missing_files`


In [2]:
def build_synthetic_db(db_path):
    """Create a small djuced.db-shaped SQLite fixture for offline use of this guide."""
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE tracks (
            id INTEGER PRIMARY KEY, artist TEXT, title TEXT, absolutepath TEXT,
            genre TEXT, bpm REAL, rating INTEGER, playcount INTEGER,
            last_played TEXT, waveform BLOB
        )
    """)
    cur.execute("""
        CREATE TABLE playlists2 (
            name TEXT, path TEXT, data TEXT, order_in_list INTEGER, type INTEGER
        )
    """)
    cur.execute("""
        CREATE TABLE trackCues (
            id INTEGER PRIMARY KEY, trackId TEXT, cuename TEXT, cuenumber INTEGER,
            cuepos REAL, loopLength REAL, cueColor INTEGER, isSavedLoop INTEGER
        )
    """)
    cur.execute("""
        CREATE TABLE trackBeats (
            id INTEGER PRIMARY KEY, trackId TEXT, beatpos BLOB,
            timesignature INTEGER, downbeat INTEGER, grid INTEGER, auftakt TEXT
        )
    """)
    cur.execute("CREATE TABLE recordings (id INTEGER PRIMARY KEY, recordId TEXT)")

    fixtures_dir = db_path.parent / "fixture_audio"
    fixtures_dir.mkdir(exist_ok=True)

    # -- library-browsing cluster: gaps of ~15-20s, too fast to be real mixing --
    browsing = [
        (1, "Demo Artist A", "Demo Track A", "2024-01-06T14:00:00"),
        (2, "Demo Artist B", "Demo Track B", "2024-01-06T14:00:15"),
        (3, "Demo Artist C", "Demo Track C", "2024-01-06T14:00:32"),
    ]

    # -- real mixing session: ~4 minutes apart, matches typical track length --
    mixing = [
        (4, "Demo Artist D", "Demo Track D", "2024-01-06T14:20:00"),
        (5, "Demo Artist E", "Demo Track E", "2024-01-06T14:24:10"),
        (6, "Demo Artist F", "Demo Track F", "2024-01-06T14:28:05"),
        (7, "Demo Artist G", "Demo Track G", "2024-01-06T14:32:20"),
        (8, "Demo Artist H", "Demo Track H", "2024-01-06T14:36:00"),
        (9, "Demo Artist I", "Demo Track I", "2024-01-06T14:40:16"),
    ]

    # -- decoy session: similar start-of-day time, very different (short) duration --
    decoy = [
        (10, "Demo Artist J", "Decoy Track J", "2024-02-10T14:05:00"),
        (11, "Demo Artist K", "Decoy Track K", "2024-02-10T14:07:00"),
    ]

    all_tracks = browsing + mixing + decoy
    for track_id, artist, title, last_played in all_tracks:
        if track_id == 3:
            # deliberately missing file, for find_missing_files()
            path = str(fixtures_dir / "does_not_exist_track_3.mp3")
        else:
            path = str(fixtures_dir / f"track_{track_id}.mp3")
            Path(path).touch()
        cur.execute(
            "INSERT INTO tracks (id, artist, title, absolutepath, genre, bpm, rating, "
            "playcount, last_played, waveform) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (track_id, artist, title, path, "Demo Techno", 128.0, 4,
             track_id % 5, last_played, None),
        )

    mixing_paths = [str(fixtures_dir / f"track_{tid}.mp3") for tid, *_ in mixing]
    cur.execute(
        "INSERT INTO playlists2 (name, path, data, order_in_list, type) VALUES (?, ?, ?, ?, ?)",
        ("Warmup Set", "#", "", 0, 0),
    )
    for i, path in enumerate(mixing_paths, start=1):
        cur.execute(
            "INSERT INTO playlists2 (name, path, data, order_in_list, type) VALUES (?, ?, ?, ?, ?)",
            ("Warmup Set", "#", path, i, 3),
        )
    cur.execute(
        "INSERT INTO playlists2 (name, path, data, order_in_list, type) VALUES (?, ?, ?, ?, ?)",
        ("AllSongsUnalyzed", "#", "", 0, 5),
    )

    cue_track = mixing_paths[0]
    for cuenum, cuepos in enumerate([0.5, 30.2, 90.0]):
        cur.execute(
            "INSERT INTO trackCues (trackId, cuename, cuenumber, cuepos, loopLength, "
            "cueColor, isSavedLoop) VALUES (?, ?, ?, ?, ?, ?, ?)",
            (cue_track, f"Cue {cuenum}", cuenum, cuepos, 0, 4, 0),
        )

    for tid, path in zip([t[0] for t in mixing], mixing_paths):
        cur.execute(
            "INSERT INTO trackBeats (trackId, beatpos, timesignature, downbeat, grid, auftakt) "
            "VALUES (?, ?, ?, ?, ?, ?)",
            (path, f"{(tid % 10) / 100:.2f}".encode(), 4, 0, 1, None),
        )

    # recording: time-of-day close to BOTH mixing (14:20) and decoy (14:05) session starts,
    # but its ~20-minute duration only matches the mixing session
    cur.execute(
        "INSERT INTO recordings (recordId) VALUES (?)",
        ("C:/fixture/Records/My Mix - 14h20m49s to 14h40m16s.mp3",),
    )

    conn.commit()
    conn.close()


if lm.DEFAULT_DB_PATH.exists():
    db_path = lm.DEFAULT_DB_PATH
    source = f"real DJUCED library ({db_path})"
else:
    synthetic_dir = Path(os.environ.get("TEMP", "/tmp")) / "live_mixing_guide_fixture"
    synthetic_dir.mkdir(exist_ok=True)
    db_path = synthetic_dir / "djuced.db"
    if not db_path.exists():
        build_synthetic_db(db_path)
    source = f"SYNTHETIC (no djuced.db found) — fixture at {db_path}"

tracks_preview = lm.read_djuced_db(db_path=db_path)
print(f"Data source : {source}")
print(f"Tracks      : {len(tracks_preview)} rows, {tracks_preview.shape[1]} columns")
tracks_preview[["artist", "title", "playcount", "last_played"]].head()


Data source : real DJUCED library (C:\Users\l_ace\Documents\DJUCED\djuced.db)
Tracks      : 1603 rows, 34 columns


,artist,title,playcount,last_played
0,Loopmasters,Gun Shot,0,None
1,Loopmasters,Noize Hit,0,None
2,Loopmasters,Closed Hat,0,None
3,Loopmasters,Kick,0,None
4,Loopmasters,Air Horn TJ,0,None


## 2. Core reader — `read_djuced_db`

`read_djuced_db(db_path, table="tracks", query=None)` is the base helper every other `read_*`
function goes through. Use `table=` for a plain `SELECT *`, or `query=` for custom SQL — every
join-based reader in this package (`read_djuced_playlist_tracks`, `read_track_cues`, ...) is just
a `query=` call under the hood.


In [3]:
# table= mode: a plain SELECT * FROM <table>
recordings = lm.read_djuced_db(db_path=db_path, table="recordings")
print(f"recordings table: {len(recordings)} rows")
recordings.head()


recordings table: 21 rows


,id,recordId
0,1,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...
1,2,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...
2,3,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...
3,4,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...
4,5,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...


In [4]:
# query= mode: any custom SQL — here, the 5 highest-BPM tracks
fastest = lm.read_djuced_db(
    db_path=db_path,
    query="SELECT artist, title, bpm FROM tracks WHERE bpm IS NOT NULL ORDER BY bpm DESC LIMIT 5",
)
print(f"fastest tracks: {len(fastest)} rows")
fastest


fastest tracks: 5 rows


,artist,title,bpm
0,Plastikman,Track 04,144.757004
1,artist,Hardfloor - 06_Into the Nature (Plastikman Mix),138.873993
2,artist,Hardfloor - 06_Into the Nature (Plastikman Mix),138.873993
3,DJ Rolando,Knights of the Jaguar (Original Mix),137.681000
4,Ricardo Villalobos,Logohitz,136.000000


## 3. Playlists — `read_djuced_playlists`, `read_djuced_playlist_tracks`

`playlists2` is a mixed-purpose table (see `docs/schemas.md`): `type=0` rows are playlist/crate
name headers, `type=3` rows are actual track entries (their `data` column holds the track's file
path — the same join key, `tracks.absolutepath`, used everywhere in this package), and `type=5` is
a single "AllSongsUnalyzed" marker row. `read_djuced_playlist_tracks` does the `type=3` join for
you.


In [5]:
playlists = lm.read_djuced_playlists(db_path=db_path)
print(f"playlists2: {len(playlists)} rows")
print("type counts:")
print(playlists['type'].value_counts().sort_index())
playlists.head()


playlists2: 189 rows
type counts:
type
0     10
3    178
5      1
Name: count, dtype: int64


,name,path,data,order_in_list,type
0,AllSongsUnalyzed,#,,0,5
1,build,#,,1,0
2,warmup,#,,2,0
3,peak,#,,3,0
4,breakdown,#,,4,0


In [6]:
playlist_tracks = lm.read_djuced_playlist_tracks(db_path=db_path)
print(f"playlist x track rows: {len(playlist_tracks)}")
playlist_tracks[["playlist_name", "order_in_list", "artist", "title"]].head(10)


playlist x track rows: 178


,playlist_name,order_in_list,artist,title
0,20260804,1,Dop,Ikarus
1,20260804,2,Andy Lee,Lope
2,20260804,3,Heartthrob,Futures Past (Original Mix)
3,20260804,4,Martinez,Momomowha
4,20260804,5,Daso,Thujon
5,20260804,6,Reboot,Ronson (Original Mix)
6,20260804,7,Teflon,The Wombat
7,20260804,8,TG aka Tim Green,Mr Dry (Dachshund Bottomless Remix)
8,20260804,9,Mikael Stavostrand,Moment And Movement
9,20260804,10,Elektrochemie,Mucky Star


## 4. Track detail — `read_track_cues`, `read_track_beatgrid`

Both join on `tracks.absolutepath` (not `tracks.id` — see the join-key convention in
`docs/schemas.md`). Omit `track_absolutepath` to get cues/beatgrid rows for the **whole**
library; pass one to scope to a single track. `beatpos` in `read_track_beatgrid` comes back as
raw bytes — DJUCED's packed beatgrid format is undocumented, so this package returns it as-is
rather than guessing a decoding.


In [7]:
# pick whichever track actually has cue data, so this cell works on real or synthetic data
any_cue = lm.read_track_cues(db_path=db_path)
if any_cue.empty:
    print("No cues in this database.")
else:
    sample_track = any_cue.iloc[0]
    print(f"Showing cues for: {sample_track['artist']} - {sample_track['title']}")
    display_cols = ["artist", "title", "cuename", "cuenumber", "cuepos", "isSavedLoop"]
    print(any_cue[any_cue["title"] == sample_track["title"]][display_cols])


Showing cues for:  - $ingoli Electro Minimal - Matt John - Io
   artist                                     title cuename  cuenumber  \
0          $ingoli Electro Minimal - Matt John - Io   Cue 0          0   
1          $ingoli Electro Minimal - Matt John - Io       1       1000   
2          $ingoli Electro Minimal - Matt John - Io       3       1001   
3          $ingoli Electro Minimal - Matt John - Io       4       1002   
4          $ingoli Electro Minimal - Matt John - Io       3       1003   
5          $ingoli Electro Minimal - Matt John - Io       2       1004   
6          $ingoli Electro Minimal - Matt John - Io       3       1005   
7          $ingoli Electro Minimal - Matt John - Io       4       1006   
8          $ingoli Electro Minimal - Matt John - Io       3       1007   
9          $ingoli Electro Minimal - Matt John - Io       5       1008   
10         $ingoli Electro Minimal - Matt John - Io       7       1009   
11         $ingoli Electro Minimal - Matt John - I

In [8]:
beatgrid = lm.read_track_beatgrid(db_path=db_path)
print(f"beatgrid rows: {len(beatgrid)}")
beatgrid[["artist", "title", "timesignature", "downbeat", "grid", "beatpos"]].head()


beatgrid rows: 1602


,artist,title,timesignature,downbeat,grid,beatpos
0,,$ingoli Electro Minimal - Matt John - Io,14,NaN,NaN,b'0'
1,,$ingoli Electro Minimal - Matt John - Io,3997696,NaN,NaN,b'0.215827664399093'
2,,-Sw.Sr.--[2006.11]--7 Grindvik - Do Us Apart,14,NaN,NaN,b'0'
3,,-Sw.Sr.--[2006.11]--7 Grindvik - Do Us Apart,35,NaN,NaN,b'0.213537414965986'
4,,01 - Atomium - Joris Vermeiren,7209071,NaN,NaN,b'0'


## 5. Session reconstruction — `read_djuced_session`, `list_sessions`

DJUCED has no session/play-log table. `list_sessions(gap_minutes=15)` reconstructs session
boundaries by clustering `tracks.last_played` timestamps: a gap larger than `gap_minutes` starts a
new session. 15 minutes was empirically tuned against real DJ history to separate back-to-back
mixing (gaps of tens of seconds to a few minutes) from library browsing/cueing (gaps of only a
few seconds) — see `docs/architecture.md` before changing it.

`read_djuced_session(start, end)` is the lower-level building block: given a timestamp range, it
returns the tracks played in it. `list_sessions()` uses the same idea across the *whole* history
to find the ranges automatically.

**Caveat**: `last_played` stores only the most recent play per track — a track replayed after the
window you're querying won't show up for an earlier session. This is a real limitation of the
data, not a bug.


In [9]:
sessions = lm.list_sessions(db_path=db_path, gap_minutes=GAP_MINUTES)
print(f"Detected sessions: {len(sessions)}")
sessions.sort_values("n_tracks", ascending=False).head(10)


Detected sessions: 96


,session_id,date,start,end,n_tracks,duration_min
37,37,2026-07-29,2026-07-29 12:34:29,2026-07-29 13:13:38,37,39.2
90,90,2026-08-15,2026-08-15 14:57:26,2026-08-15 17:06:11,32,128.8
92,92,2026-08-16,2026-08-16 14:39:10,2026-08-16 16:01:16,25,82.1
67,67,2026-08-05,2026-08-05 14:32:48,2026-08-05 14:41:11,25,8.4
52,52,2026-08-03,2026-08-03 08:14:54,2026-08-03 08:34:08,22,19.2
24,24,2026-07-23,2026-07-23 13:23:31,2026-07-23 13:42:17,18,18.8
66,66,2026-08-04,2026-08-04 18:42:02,2026-08-04 19:58:21,17,76.3
13,13,2026-07-10,2026-07-10 12:02:23,2026-07-10 13:28:29,16,86.1
23,23,2026-07-23,2026-07-23 10:54:31,2026-07-23 11:54:31,16,60.0
30,30,2026-07-25,2026-07-25 22:21:43,2026-07-25 23:08:08,15,46.4


In [10]:
# drill into the session with the most tracks, using read_djuced_session directly
biggest = sessions.sort_values("n_tracks", ascending=False).iloc[0]
print(f"Session {biggest['session_id']} on {biggest['date']}: "
      f"{biggest['n_tracks']} tracks over {biggest['duration_min']} min")

setlist = lm.read_djuced_session(
    db_path=db_path, start=biggest["start"].isoformat(), end=biggest["end"].isoformat()
)
setlist[["last_played", "artist", "title"]]


Session 37 on 2026-07-29: 37 tracks over 39.2 min


,last_played,artist,title
0,2026-07-29T12:34:29,SIS,Nu Wim De Wa (Original Mix)
1,2026-07-29T12:34:47,,Tom Budden - The Tree Dance (O
2,2026-07-29T12:34:59,Jon Seber,Speechless Monkeys (Delete Remix)
3,2026-07-29T12:35:09,Delete,Flight Schedule (Gregor Tresher Remix)
4,2026-07-29T12:37:09,Andre Kraml,Safari (James Holden Remix)
5,2026-07-29T12:51:52,Agaric,Goose Step
6,2026-07-29T12:52:38,Alex Under,El Diluvio Azul (Original Mix)
7,2026-07-29T12:52:42,Alexi Delano & Xpansul,Echolocation
8,2026-07-29T12:52:58,Ambivalent,Nineteen
9,2026-07-29T12:53:43,Baffa,Sospecha sospechosa (Ilario Alicante Remix)


**Now playing**: `current_track()` is the lightest-weight of these functions — the single row with
the latest `last_played`, i.e. "what's playing right now" (the same query pattern external
now-playing pollers like [unbox](https://github.com/erikrichardlarson/unbox) use for their DJUCED
integration). It returns an empty DataFrame if no track has ever been played.


In [ ]:
now_playing = lm.current_track(db_path=db_path)
if now_playing.empty:
    print("No track has ever been played in this database.")
else:
    print(now_playing[["artist", "title", "bpm", "key", "genre", "last_played"]].to_string(index=False))


## 6. Recording ↔ session matching — `match_recording_to_session`

`recordings.recordId` filenames only encode a **time-of-day** range (e.g.
`"My Mix - 15h04m49s to 17h13m16s.mp3"`) — no date. That means two sessions on different dates can
start at a similar clock time, so matching by start-time proximity alone is unreliable (this was
tried first and produced wrong matches — see `docs/architecture.md`). Instead, candidates are
shortlisted by start-time-of-day proximity, then ranked by closeness of **duration**
(recording length vs. session span), which is a far more discriminating signal.

The `ambiguous` column flags matches where a second candidate's duration was almost as close as
the winner's — worth a manual look before trusting those rows.


In [11]:
matches = lm.match_recording_to_session(db_path=db_path, gap_minutes=GAP_MINUTES)
print(f"Recordings matched: {matches['match'].sum()}/{len(matches)}  "
      f"(ambiguous: {matches['ambiguous'].sum()})")
matches[["recording_path", "rec_start", "rec_end", "session_date", "n_tracks", "match", "ambiguous"]]


Recordings matched: 20/21  (ambiguous: 12)

,recording_path,rec_start,rec_end,session_date,n_tracks,match,ambiguous
0,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,11:04:33,11:05:39,2026-07-29,1.0,True,True
1,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,15:23:49,16:05:27,2026-07-22,7.0,True,False
2,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,11:23:14,11:24:36,2026-07-29,1.0,True,True
3,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,11:27:29,11:28:43,2026-07-29,1.0,True,True
4,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,16:31:34,16:37:35,2026-08-09,2.0,True,True
5,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,16:43:54,16:50:34,2026-08-09,2.0,True,True
6,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,11:13:12,12:08:23,2026-07-23,16.0,True,False
7,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,15:27:11,15:45:48,2026-07-30,10.0,True,True
8,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,15:59:13,16:28:18,2026-07-24,4.0,True,True
9,C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix...,15:41:44,16:25:50,2026-08-08,5.0,True,False


## 7. Play-log tracking — `snapshot_play_log`

DJUCED's `tracks.last_played` only ever stores the **most recent** play — replay a track later and
its earlier session's timestamp is gone (see Section 5's caveat and `docs/architecture.md`).
`snapshot_play_log(log_dir, db_path)` works around this by keeping its own append-only log outside
`djuced.db`: each call diffs the current `playcount` for every track against the previous snapshot
saved in `log_dir`, and appends any newly detected plays to `log_dir/play_events.csv` — once a
play is logged there, a later replay overwriting `last_played` in `djuced.db` can no longer erase
it.

Call it right after finishing a real session, before starting a new one — the shorter the gap
between calls, the more precisely each detected play can be attributed to that session.

The first snapshot below runs against your real `db_path` — **read-only**, nothing is ever written
to `djuced.db`. To show a detected play without touching your real library, the second snapshot
runs against a **throwaway copy** of the database with one track's `playcount` bumped.


In [ ]:
PLAY_LOG_DIR = Path(os.environ.get("TEMP", "/tmp")) / "live_mixing_guide_play_log_demo"
if PLAY_LOG_DIR.exists():
    shutil.rmtree(PLAY_LOG_DIR)  # start clean so this cell is idempotent on re-run

baseline_events = lm.snapshot_play_log(log_dir=PLAY_LOG_DIR, db_path=db_path)
print(f"Baseline snapshot -> new events: {len(baseline_events)} (always 0 — nothing to diff yet)")
print(f"State saved to    : {PLAY_LOG_DIR / 'play_log_state.csv'}")


In [ ]:
demo_db_path = PLAY_LOG_DIR / "demo_copy.db"
shutil.copy(db_path, demo_db_path)

demo_track_id = int(tracks_preview.iloc[0]["id"])
conn = sqlite3.connect(demo_db_path)
conn.execute("UPDATE tracks SET playcount = playcount + 1 WHERE id = ?", (demo_track_id,))
conn.commit()
conn.close()

new_events = lm.snapshot_play_log(log_dir=PLAY_LOG_DIR, db_path=demo_db_path)
print(f"Second snapshot -> new events: {len(new_events)}")
new_events[["id", "artist", "title", "playcount_before", "playcount_after", "playcount_delta"]]


## 8. Library maintenance — `find_missing_files`, `top_played_tracks`

`find_missing_files` stats every `tracks.absolutepath` against disk — useful after
moving/renaming a music folder (DJUCED keeps stale references until re-scanned). `top_played_tracks`
is a simple `playcount`-ranked query.


In [12]:
missing = lm.find_missing_files(db_path=db_path)
print(f"Missing files: {len(missing)}/{len(tracks_preview)} tracks")
missing.head(10)


Missing files: 808/1603 tracks

,id,artist,title,absolutepath
0,20,Gaiser,Pullpush,D:/mUSICA/luigi/01 - Gaiser - Pullpush.mp3
1,21,Yankee Zulu,Toma! - Original Mix,D:/mUSICA/luigi/01 Yankee Zulu - Toma! (Origin...
2,22,Ambivalent,Nineteen,D:/mUSICA/luigi/01-ambivalent-nineteen (0daymu...
3,23,Ilario Alicante,Jamdishes (Original Mix),D:/mUSICA/luigi/01-ilario_alicante-jamdishes_(...
4,24,Larsson,Need the Sun,D:/mUSICA/luigi/01-larsson--need_the_sun-siber...
5,25,Marc Antona,Red Faces (Original Mix),D:/mUSICA/luigi/01-marc_antona-red_faces_(orig...
6,26,Markus Fix,Research (Original Mix),D:/mUSICA/luigi/01-markus_fix-research_(origin...
7,27,Pan-Pot,Confronted,D:/mUSICA/luigi/01-pan-pot--confronted-dh.mp3
8,28,Ramon Tapia,Say What (Original Mix),D:/mUSICA/luigi/01-ramon_tapia-say_what_(origi...
9,29,Ruede Hagelstein Feat Naneci,Semikolon (Original Mix),D:/mUSICA/luigi/01-ruede_hagelstein_feat_nanec...


In [13]:
top = lm.top_played_tracks(n=10, db_path=db_path)
top


,artist,title,playcount,last_played
0,Paco Osuna,Alarm (Original Mix),27,2026-08-10T08:15:26
1,Marek Bois,B1 You Got Good Ash (Gabriel Ananda Remix),25,2026-08-15T14:33:49
2,Ambivalent,R U OK (Marco Carola 2 Beats remix),24,2026-08-15T14:28:49
3,Spencer Parker & Diesel pres Co.Lab,Zanzibar (Nick Curly Mix),22,2026-08-15T15:56:35
4,Click Box,Bass Tilt,20,2026-08-16T15:56:53
5,Marc houle,East to west,19,2026-07-21T18:35:11
6,2000 and One,2000_and_one-pak_pak - pure037_,19,2026-08-03T23:46:26
7,Delete,Let George Do It,17,2026-07-22T10:03:37
8,Argy,Tronic Jams (The Storm),17,2026-08-10T08:49:26
9,Koljah,My Room (Original Mix),16,2026-07-22T09:58:25


## 9. CSV export

All five `export_*_csv` functions write to disk (here, under `data/`) and also return the
DataFrame they wrote, so you can inspect the result without re-reading the file.
`export_tracks_csv` drops the `waveform` blob column by default — it's an ~8KB binary blob per
row, not meaningful as CSV text. `export_session_setlist_csv` looks up a `session_id` from
`list_sessions()` and raises `ValueError` if it doesn't exist (see the error-handling section
below).


In [14]:
playlists_csv = lm.export_playlists_csv(str(OUTPUT_DIR / "playlists_export.csv"), db_path=db_path)
sessions_csv = lm.export_sessions_csv(str(OUTPUT_DIR / "sessions_export.csv"), db_path=db_path, gap_minutes=GAP_MINUTES)
setlist_csv = lm.export_session_setlist_csv(
    int(biggest["session_id"]), csv_path=str(OUTPUT_DIR / "setlist_biggest_session.csv"), db_path=db_path
)
print(f"Wrote playlists_export.csv  : {len(playlists_csv)} rows")
print(f"Wrote sessions_export.csv   : {len(sessions_csv)} rows")
print(f"Wrote setlist_biggest_session.csv : {len(setlist_csv)} rows")


Wrote playlists_export.csv  : 189 rows


Wrote sessions_export.csv   : 96 rows
Wrote setlist_biggest_session.csv : 37 rows


In [15]:
if RUN_FULL:
    tracks_csv = lm.export_tracks_csv(str(OUTPUT_DIR / "tracks_export.csv"), db_path=db_path)
    print(f"Wrote tracks_export.csv: {len(tracks_csv)} rows")
else:
    print("RUN_FULL is False — skipping the full tracks-table export (set RUN_FULL = True above to run it).")


RUN_FULL is False — skipping the full tracks-table export (set RUN_FULL = True above to run it).


## 10. End-to-end pattern

A realistic chain: pick the largest session that also has a matched recording (falling back to
the overall busiest session if none matched), pull track detail (cues) for the first track
played, confirm the recording match, and export the setlist — the same flow used to answer "what
did I actually play, and do I have a recording of it?" for a real DJUCED library.


In [16]:
matched_session_ids = set(matches.loc[matches["match"], "session_id"].dropna().astype(int))
matched_sessions = sessions[sessions["session_id"].isin(matched_session_ids)]

if not matched_sessions.empty:
    target = matched_sessions.sort_values("n_tracks", ascending=False).iloc[0]
else:
    target = biggest  # fallback: no recording matched any session

print(f"Target session  : {target['session_id']} on {target['date']} "
      f"({target['n_tracks']} tracks, {target['duration_min']} min)")

target_setlist = lm.read_djuced_session(
    db_path=db_path, start=target["start"].isoformat(), end=target["end"].isoformat()
)
first_track = target_setlist.iloc[0]
print(f"First track     : {first_track['artist']} - {first_track['title']}")

first_track_cues = lm.read_track_cues(db_path=db_path, track_absolutepath=first_track["absolutepath"])
print(f"Hot cues on it  : {len(first_track_cues)}")

session_matches = matches[matches["session_id"] == target["session_id"]]
if session_matches.empty:
    print("Recording match : none found for this session")
else:
    print(f"Recording match : {session_matches.iloc[0]['recording_path']}")

exported = lm.export_session_setlist_csv(
    int(target["session_id"]), csv_path=str(OUTPUT_DIR / "setlist_end_to_end.csv"), db_path=db_path
)
print(f"Exported        : {len(exported)}-track setlist -> data/setlist_end_to_end.csv")


Target session  : 90 on 2026-08-15 (32 tracks, 128.8 min)
First track     : Robag Wruhme - Ratibor Numida (Original Mix)
Hot cues on it  : 16
Recording match : C:/Users/l_ace/Music/DJUCED/Records/Mix/My Mix - 15h04m49s to 17h13m16s.mp3
Exported        : 32-track setlist -> data/setlist_end_to_end.csv


## 11. Error handling reference

| Call | Guard | Outcome |
|---|---|---|
| `read_djuced_db(db_path="bad/path.db")` | `db_path.exists()` check | **raises** `FileNotFoundError` |
| `export_session_setlist_csv(session_id=<unknown>)` | looked up against `list_sessions()` | **raises** `ValueError` |
| `read_track_cues(track_absolutepath=<unknown>)` | SQL `WHERE` filter matches nothing | **degrades** — returns an empty `DataFrame`, no error |
| `read_track_beatgrid(track_absolutepath=<unknown>)` | same | **degrades** — empty `DataFrame` |
| `match_recording_to_session()` on a `recordings` row whose filename doesn't match the expected pattern | regex `search` returns `None` | **degrades** — that row gets `match=False`, rest still processed |
| `find_missing_files()` on a track whose drive/volume isn't currently mounted | `Path.exists()` | **degrades**, but note: an unmounted (not just deleted) drive will show *all* its tracks as "missing" even though the files may still be fine once reconnected |
| `snapshot_play_log()` when a track's `playcount` is *lower* than the last snapshot (db reset/restore) | diff only fires on an increase | **degrades** — no event is fabricated, that track is silently rebaselined to the current (lower) value |

The first two are setup/usage mistakes and are meant to stop you immediately. The rest reflect
real, expected gaps in DJUCED's own data (an unanalyzed track has no cues; not every recording's
filename parses; not every track has a cue point) and are designed to degrade quietly so a single
bad row doesn't blow up a whole-library query.


In [17]:
try:
    lm.read_djuced_db(db_path="this/path/does/not/exist.db")
except FileNotFoundError as e:
    print(f"FileNotFoundError (expected): {e}")

try:
    lm.export_session_setlist_csv(session_id=999999, db_path=db_path)
except ValueError as e:
    print(f"ValueError (expected): {e}")

empty_cues = lm.read_track_cues(db_path=db_path, track_absolutepath="no/such/track.mp3")
print(f"Unknown track cues -> degrades to empty DataFrame: {len(empty_cues)} rows, no exception raised")


FileNotFoundError (expected): DJUCED database not found: this\path\does\not\exist.db
ValueError (expected): No session with session_id=999999
Unknown track cues -> degrades to empty DataFrame: 0 rows, no exception raised
